# LoRA SFT + DPO fine-tuning (Unsloth, free T4)

Run this on Colab with a free T4 GPU (Runtime > Change runtime type > T4 GPU).
Clones `voice-agent-eval-bench`, installs Unsloth, runs LoRA SFT on
`data/finetune/sft.jsonl` then DPO on `data/finetune/prefs.jsonl`, on top of
Qwen2.5-1.5B-Instruct (or Llama-3.2-1B-Instruct if Unsloth's 4-bit build is
more current for that model — check https://huggingface.co/unsloth for the
latest 4-bit repo names before running).

Reports actual training time and peak GPU memory — that's Unsloth's real
selling point on a free T4, so this notebook measures it rather than just
naming the library.

In [ ]:
!git clone https://github.com/saitejasrivilli/voice-agent-eval-bench.git
%cd voice-agent-eval-bench
!pip install -q unsloth

In [ ]:
import json, time, torch
from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit"  # verify current 4-bit repo name on huggingface.co/unsloth before running
MAX_SEQ_LEN = 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

In [ ]:
# --- SFT step ---
# Load data/finetune/sft.jsonl, format as chat turns, train with TRL's SFTTrainer
# per Unsloth's documented pattern. Adapt hyperparameters (batch size, epochs,
# learning rate) from the reference rlhf-synthesis-optimization/sft_warmup.py
# script (paste its contents in before running this cell).

torch.cuda.reset_peak_memory_stats()
sft_start = time.time()

# ... SFTTrainer setup + .train() goes here, following sft_warmup.py ...

sft_time_sec = time.time() - sft_start
sft_peak_mem_gb = torch.cuda.max_memory_allocated() / 1e9
print(f"SFT training time: {sft_time_sec:.1f}s, peak GPU memory: {sft_peak_mem_gb:.2f} GB")

In [ ]:
# --- DPO step ---
# Load data/finetune/prefs.jsonl, train with TRL's DPOTrainer on top of the
# SFT checkpoint, per Unsloth's documented drop-in pattern. Adapt hyperparameters
# from the reference rlhf-synthesis-optimization/train_dpo.py script.

torch.cuda.reset_peak_memory_stats()
dpo_start = time.time()

# ... DPOTrainer setup + .train() goes here, following train_dpo.py ...

dpo_time_sec = time.time() - dpo_start
dpo_peak_mem_gb = torch.cuda.max_memory_allocated() / 1e9
print(f"DPO training time: {dpo_time_sec:.1f}s, peak GPU memory: {dpo_peak_mem_gb:.2f} GB")

In [ ]:
import os, shutil
os.makedirs("checkpoints", exist_ok=True)
model.save_pretrained("checkpoints/lora_adapter")
tokenizer.save_pretrained("checkpoints/lora_adapter")
shutil.make_archive("checkpoints/lora_adapter", "zip", "checkpoints/lora_adapter")

with open("checkpoints/training_stats.json", "w") as f:
    json.dump({
        "sft_time_sec": sft_time_sec,
        "sft_peak_mem_gb": sft_peak_mem_gb,
        "dpo_time_sec": dpo_time_sec,
        "dpo_peak_mem_gb": dpo_peak_mem_gb,
    }, f, indent=2)

from google.colab import files
files.download("checkpoints/lora_adapter.zip")
files.download("checkpoints/training_stats.json")